In [1]:
import sys
import os

# Add the parent directory (src) to the system path
# The '..' tells it to look one folder up from where the notebook is currently running
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
#Loadin the rag
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [3]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [4]:
#Judge Instructions
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [5]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [9]:
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress
from client import client


In [8]:
#check a record
rec = answers[0]
rec

{'question': 'Is it too late to enroll in the course if I just found out about it?',
 'answer_llm': 'Yes, you can still join the course, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [10]:
#Evaluation of the answers using the LLM as a judge
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-4o-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [12]:
#Test it on one record
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer accurately reflects the key points of the original answer. It states that it is possible to join the course but emphasizes the requirement to submit the project on time for certification, which aligns with the original intent. There are no critical omissions or incorrect statements.', score='good')

In [13]:
#run it for all
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [14]:
#parallelize the evaluation for all records
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/515 [00:00<?, ?it/s]

APIStatusError: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 16384 tokens, but can only afford 16227. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None, 'previous_errors': [{'code': 402, 'message': 'This request requires more credits, or fewer max_tokens. You requested up to 16384 tokens, but can only afford 16227. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account'}]}}, 'user_id': 'user_3GFHKSIHSEC3Klpq5qteFqrd6OM'}

In [15]:
#Split the results and usage
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

NameError: name 'results' is not defined

In [16]:
#create a dataframe for the evaluations
df_eval = pd.DataFrame(evaluations)

In [17]:
calc_total_price(usages)

0.0

In [18]:
#Check the result
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

KeyError: 'score'

In [19]:
#Look at the bad cases
df_eval[df_eval["score"] == "bad"].head()

KeyError: 'score'

In [21]:
#save the results
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)